In [1]:
# ============================================================
# 🏛️ Supreme Court Rich Metadata Extractor
# Parses metadata/*.json -> rich_metadata_2020_2025.csv
# Author: LPA Project
# ============================================================


In [2]:
import os
import json
import re
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

In [3]:
# ---------- CONFIG ----------
BASE_DIR = r"D:\LPA_MTech_Project\My_Datasets\SC_2020-2025"
OUTPUT_CSV = os.path.join(BASE_DIR, "rich_metadata_2020_2025.csv")
START_YEAR = 2020
END_YEAR = 2025
# ----------------------------
records = []

In [4]:
def extract_case_details(raw_html):
    """Extract details from the raw_html field of metadata JSON."""
    soup = BeautifulSoup(raw_html, "html.parser")

    # --- Case name ---
    case_name = ""
    case_btn = soup.find("button", {"aria-label": True})
    if case_btn and "aria-label" in case_btn.attrs:
        case_name = (
            case_btn["aria-label"]
            .replace(" pdf", "")
            .strip()
        )

    # --- Coram / Judges ---
    coram = ""
    coram_match = re.search(r"Coram\s*:\s*(.*?)<", raw_html, re.IGNORECASE)
    if coram_match:
        coram = re.sub(r"<.*?>", "", coram_match.group(1)).strip()

    # --- Decision Date ---
    date_match = re.search(r"Decision Date\s*:\s*<\/span><font color='green'>(.*?)<\/font>", raw_html)
    decision_date = date_match.group(1).strip() if date_match else ""

    # --- Case No ---
    case_no_match = re.search(r"Case No\s*:\s*<\/span><font color='green'>(.*?)<\/font>", raw_html)
    case_no = case_no_match.group(1).strip() if case_no_match else ""

    # --- Disposal Nature ---
    disp_match = re.search(r"Disposal Nature\s*:\s*<\/span><font color='green'>(.*?)<\/font>", raw_html)
    disposal = disp_match.group(1).strip() if disp_match else ""

    # --- Bench ---
    bench_match = re.search(r"Bench\s*:\s*<\/span><font color='green'>(.*?)<\/font>", raw_html)
    bench = bench_match.group(1).strip() if bench_match else ""

    # --- Short summary ---
    # After </strong><br> ... text before next <strong>
    summary = ""
    summary_match = re.search(r"</strong><br>(.*?)<strong class='caseDetailsTD'>", raw_html, re.DOTALL)
    if summary_match:
        summary = (
            BeautifulSoup(summary_match.group(1), "html.parser")
            .get_text(separator=" ", strip=True)
            .replace("\n", " ")
        )

    return {
        "case_name": case_name,
        "judges": coram,
        "decision_date": decision_date,
        "case_no": case_no,
        "disposal_nature": disposal,
        "bench": bench,
        "summary": summary
    }


In [5]:
# ============================================================
# 🔍 MAIN EXTRACTION LOOP
# ============================================================

for year in range(START_YEAR, END_YEAR + 1):
    meta_dir = os.path.join(BASE_DIR, str(year), "metadata")
    if not os.path.exists(meta_dir):
        continue

    print(f"\n📂 Processing metadata for {year}")
    for file in tqdm(os.listdir(meta_dir), desc=f"Year {year}", unit="file"):
        if not file.endswith(".json"):
            continue

        file_path = os.path.join(meta_dir, file)
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            raw_html = data.get("raw_html", "")
            if not raw_html.strip():
                continue

            details = extract_case_details(raw_html)
            record = {
                "year": year,
                "file_name": file,
                "path": data.get("path", ""),
                "citation_year": data.get("citation_year", ""),
                "nc_display": data.get("nc_display", ""),
                **details
            }
            records.append(record)

        except Exception as e:
            print(f"⚠️ Error reading {file}: {e}")


📂 Processing metadata for 2020


Year 2020: 100%|██████████| 571/571 [00:01<00:00, 363.78file/s]



📂 Processing metadata for 2021


Year 2021: 100%|██████████| 708/708 [00:01<00:00, 409.53file/s]



📂 Processing metadata for 2022


...

Year 2023: 100%|██████████| 856/856 [00:02<00:00, 342.58file/s]



📂 Processing metadata for 2024


Year 2024: 100%|██████████| 782/782 [00:02<00:00, 352.59file/s]



📂 Processing metadata for 2025


Year 2025: 100%|██████████| 436/436 [00:01<00:00, 372.99file/s]


In [6]:
# ============================================================
# 💾 SAVE TO CSV
# ============================================================

df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"\n✅ Rich metadata CSV created successfully: {OUTPUT_CSV}")
print(f"Total cases processed: {len(df)}")
df.head(10)


✅ Rich metadata CSV created successfully: D:\LPA_MTech_Project\My_Datasets\SC_2020-2025\rich_metadata_2020_2025.csv
Total cases processed: 4370


,year,file_name,path,citation_year,nc_display,case_name,judges,decision_date,case_no,disposal_nature,bench,summary
0,2020,2020_10_1043_1074.json,2020_10_1043_1074,2020,2020INSC525,Close,A.M. KHANWILKAR,01-09-2020,CIVIL APPEAL No. 7157/2008,Dismissed,2 Judges,
1,2020,2020_10_1075_1131.json,2020_10_1075_1131,2020,2020INSC531,Close,R.F. NARIMAN,02-09-2020,CRIMINAL APPEAL No. 546/2017,Disposed off,2 Judges,
2,2020,2020_10_1132_1150.json,2020_10_1132_1150,2020,2020INSC535,Close,ASHOK BHUSHAN,07-09-2020,CIVIL APPEAL No. 3093/2020,Case Partly allowed,2 Judges,
3,2020,2020_10_135_237.json,2020_10_135_237,2020,2020INSC487,Close,ARUN MISHRA,11-08-2020,DIARYNO AND DIARYYR No. 32601/2018,Directions issued,3 Judges,
4,2020,2020_10_1_26.json,2020_10_1_26,2020,2020INSC379,Close,UDAY UMESH LALIT,29-04-2020,CIVIL APPEAL No. 2379/2020,Appeal(s) allowed,2 Judges,
5,2020,2020_10_238_272.json,2020_10_238_272,2020,2020INSC539,Close,ASHOK BHUSHAN,09-09-2020,CIVIL APPEAL No. 1789/2020,Dismissed,2 Judges,
6,2020,2020_10_273_298.json,2020_10_273_298,2020,2020INSC597,Close,S.A. BOBDE,14-10-2020,CIVIL APPEAL No. 3438/2020,Disposed off,3 Judges,
7,2020,2020_10_27_134.json,2020_10_27_134,2020,2020INSC412,Close,R.F. NARIMAN,03-06-2020,CRIMINAL APPEAL No. 403/2010,Disposed off,3 Judges,
8,2020,2020_10_299_362.json,2020_10_299_362,2020,2020INSC557,Close,ASHOK BHUSHAN,21-09-2020,WRIT PETITION (CIVIL) No. 1030/2020,Disposed off,3 Judges,
9,2020,2020_10_363_374.json,2020_10_363_374,2020,2020INSC615,Close,D.Y. CHANDRACHUD,28-10-2020,CIVIL APPEAL No. 3544/2020,Disposed off,3 Judges,
